<a href="https://colab.research.google.com/github/Dmitry-Lipatov/fullstackanalyst/blob/main/1_first%20case_from%20multi%20CSV%20files%20into%20one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 кейс

**Задача написать функцию `process_files`, которая принимает на вход два пути к папкам. Из первой папки необходимо выбрать все "чеки" (файлы по шаблону из условия), а во вторую папку сохранить один объединенный чек (отсортированный по дате, а затем по продукту) под названием `combined_data.csv`.**

**Важно**

Перед началом выполним следующую ячейку, чтобы загрузить папку с файлами. После выполнения, в папке `reports_main` будут храниться все присланные магазинами чеки.

In [11]:
!wget https://github.com/vs8th/reports/archive/main.zip

import zipfile

with zipfile.ZipFile("main.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")

!rm main.zip

--2026-06-13 13:05:14--  https://github.com/vs8th/reports/archive/main.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/Vs8th/reports/zip/refs/heads/main [following]
--2026-06-13 13:05:15--  https://codeload.github.com/Vs8th/reports/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 20.27.177.114
Connecting to codeload.github.com (codeload.github.com)|20.27.177.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘main.zip’

main.zip                [ <=>                ]   3.22K  --.-KB/s    in 0s      

2026-06-13 13:05:15 (53.0 MB/s) - ‘main.zip’ saved [3296]



Чтобы посмотреть как выглядят подходящие для объединения чеки выполните следующую ячейку.

In [13]:
import pandas as pd

df = pd.read_csv('reports-main/2023-02-17-05-38-2.csv', sep=";")
df

,date,product,store,cost
0,2023-02-17,product_0,store_2,10
1,2023-02-17,product_1,store_2,20
2,2023-02-17,product_2,store_2,30
3,2023-02-17,product_3,store_2,40
4,2023-02-17,product_4,store_2,50


**Решение**

Напишите свое решение ниже

**Примечание**

Не все файлы подходящие по названию, будут подходить по содержанию. Там может оказаться лишний столбец, например. Ориентируйтесь на столбцы из чека выше - это то, что вас интересует. Остальные столбцы можно просто отбросить.

**Важно**: разделителем файла на выходе должна быть запятая.

In [14]:
import csv
import glob
import os
import re


def process_files(src_folder, dest_folder):
    """Объединяет CSV-чеки из src_folder, имя которых строго соответствует шаблону

    ГГГГ-ММ-ДД-ЧЧ-ММ-idмагазина.csv, сортирует по дате и продукту, и сохраняет
    результат в dest_folder/combined_data.csv.
    """

    def get_sort_key(row):
        # Теперь 'date' имеет индекс 0, а 'product' — индекс 1
        return (row[0], row[1])

    # 1. Ищем все CSV-файлы в исходной папке
    search_pattern = os.path.join(src_folder, "*.csv")
    file_list = glob.glob(search_pattern)

    if not file_list:
        print("Файлы не найдены.")
        return

    # Шаблон регулярного выражения из вашего условия
    pattern = re.compile(r"\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d+.csv")

    # Жестко фиксированный эталонный заголовок на выходе (без index)
    target_headers = ["date", "product", "store", "cost"]

    all_rows = []

    # 2. Читаем данные только из подходящих по шаблону файлов
    for file_path in file_list:
        file_name = os.path.basename(file_path)

        # Если имя файла не соответствует шаблону — пропускаем его
        if not pattern.match(file_name):
            continue

        with open(file_path, mode="r", encoding="utf-8") as f:
            # Читаем первую строку для определения разделителя
            first_line = f.readline()
            f.seek(0)
            if not first_line:
                continue
            current_delimiter = ";" if ";" in first_line else ","

            reader = csv.reader(f, delimiter=current_delimiter)
            try:
                file_headers = next(reader)
            except StopIteration:
                continue  # Пропускаем пустые файлы

            # Очищаем заголовки от пробелов и невидимых символов (\r, \n)
            cleaned_headers = [
                h.strip().replace("\r", "").replace("\n", "").lower()
                for h in file_headers
            ]

            # Проверяем, содержит ли файл все нужные колонки
            if not all(h in cleaned_headers for h in target_headers):
                continue

            # Находим индексы колонок во входящем файле
            date_idx = cleaned_headers.index("date")
            product_idx = cleaned_headers.index("product")
            store_idx = cleaned_headers.index("store")
            cost_idx = cleaned_headers.index("cost")

            # Собираем строки данных
            for row in reader:
                if row and len(row) >= len(file_headers):
                    # Извлекаем только нужные 4 столбца в правильном порядке
                    filtered_row = [
                        row[date_idx].strip(),
                        row[product_idx].strip(),
                        row[store_idx].strip(),
                        row[cost_idx].strip(),
                    ]
                    all_rows.append(filtered_row)

    if not all_rows:
        print("Нет подходящих данных для объединения.")
        return

    # 3. Сортируем данные с помощью именованной функции
    all_rows.sort(key=get_sort_key)

    # 4. Создаем целевую папку, если её нет
    os.makedirs(dest_folder, exist_ok=True)

    # 5. Записываем результат в итоговый файл combined_data.csv (разделитель — запятая)
    output_path = os.path.join(dest_folder, "combined_data.csv")
    with open(output_path, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter=",")
        writer.writerow(target_headers)
        writer.writerows(all_rows)

    print(f"Файл успешно сохранен: {output_path}")


src_folder = 'reports-main'
dest_folder = 'comb_reports'
process_files(src_folder, dest_folder)

Файл успешно сохранен: comb_reports/combined_data.csv


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить решение, запустим код в следующих ячейках

In [15]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv

import pandas as pd

user_answer = pd.read_csv(f'{dest_folder}/combined_data.csv')
correct_answer = pd.read_csv('data.csv')

--2026-06-13 13:05:43--  https://gist.github.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv
Resolving gist.github.com (gist.github.com)... 20.27.177.113
Connecting to gist.github.com (gist.github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv [following]
--2026-06-13 13:05:44--  https://gist.githubusercontent.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 984 [text/plain]
Saving to: ‘data.csv’

data.csv            100%[===================>]     984  --.-KB/s    in 0s      

2026-06-13 13:05:44 (77.0 MB/s) - ‘data.csv’ saved [984/984]



In [16]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Проверки пройдены успешно!')

Проверки пройдены успешно!
